# Lab 10b: A first look at PyTorch

<a href="https://colab.research.google.com/github/soonho-eric/CSC371_Fa2026/blob/main/lab11-pytorch/lab11_pytorch.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In lab 10a you wrote a multilayer perceptron by hand: the forward pass, backpropagation,
and the weight updates. In this lab you build the same network in **PyTorch**, the library
behind most deep-learning research, and train it on MNIST. Then you will play around with network size.

Run this notebook in Google Colab with a GPU. Click the badge above, then choose
*Runtime → Change runtime type → T4 GPU*.

The code follows Chapter 12 of Raschka et al., *Machine Learning with PyTorch and
Scikit-Learn*. The full chapter is in the `ch12/` folder of the course repo.

## PyTorch and training performance

Training a neural network is mostly matrix multiplication, repeated millions of times. A CPU
has a few fast cores. A GPU has thousands of slower ones, which can work on all the entries of
a matrix product at once.

<img src="https://raw.githubusercontent.com/soonho-eric/CSC371_Fa2026/main/ch12/figures/12_01.png" width="500"/>

PyTorch runs the same code on either, so you only have to say which one to use.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__)
print('Device:', device)

If this prints `cpu` in Colab, switch the runtime to a GPU (*Runtime → Change runtime type*)
and run the cell again. The notebook still works on a CPU, only more slowly.

## Tensors

A **tensor** is a multidimensional array: a scalar is a rank-0 tensor, a vector rank 1, a
matrix rank 2, and so on. A batch of 100 grayscale MNIST images is a rank-3 tensor of shape
`(100, 28, 28)`.

<img src="https://raw.githubusercontent.com/soonho-eric/CSC371_Fa2026/main/ch12/figures/12_02.png" width="400"/>

A PyTorch tensor is very much like a NumPy array, with two extras:

1. it can live on a **GPU**, and
2. it can record the operations done to it, so PyTorch can compute **gradients** automatically.

Tensors can be made from Python lists or NumPy arrays:

In [ ]:
a = [1, 2, 3]
b = np.array([4, 5, 6], dtype=np.int32)

t_a = torch.tensor(a)
t_b = torch.from_numpy(b)

print(t_a)
print(t_b)

Every tensor has a shape and a data type, and can be reshaped and transposed:

In [ ]:
t = torch.rand(3, 5)
print(t.shape, t.dtype)

t_tr = torch.transpose(t, 0, 1)
print(t.shape, ' --> ', t_tr.shape)

t_r = t.reshape(5, 3)
print(t.shape, ' --> ', t_r.shape)

The math is the same as in NumPy. `torch.multiply` is elementwise, like `*`, and
`torch.matmul` is the matrix product, like `np.dot` in lab 10.

In [ ]:
torch.manual_seed(1)
t1 = 2 * torch.rand(5, 2) - 1
t2 = torch.normal(mean=0, std=1, size=(5, 2))

t3 = torch.multiply(t1, t2)
print(t3)

t4 = torch.matmul(t1, torch.transpose(t2, 0, 1))
print(t4.shape)

`.to(device)` moves a tensor to the GPU (if there is one). Tensors in the same calculation
must be on the same device.

In [ ]:
t1_gpu = t1.to(device)
print(t1_gpu.device)

## Automatic differentiation

In lab 10 you wrote `backward` by hand. In PyTorch, a tensor created with
`requires_grad=True` records every operation applied to it. Calling `.backward()` on the loss
then applies the chain rule automatically and stores $\partial L / \partial w$ in `w.grad`.

Here is one sigmoid neuron with a squared-error loss. In lab 10 the chain rule gave
$$
\frac{\partial L}{\partial w} = 2(a - y)\cdot a(1 - a)\cdot x .
$$

In [ ]:
x = torch.tensor([1.0, 2.0])
y = torch.tensor(1.0)

w = torch.tensor([0.5, -0.4], requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

z = torch.dot(w, x) + b
a = torch.sigmoid(z)
loss = (a - y)**2

loss.backward()
print('autograd:', w.grad)

a = a.detach()   # a plain number again, without the recorded history
print('by hand: ', 2*(a - y) * a*(1 - a) * x)

## Activation functions

Each layer computes a net input $z = Wx + b$ and then applies an **activation function**
$\sigma(z)$. Without it, a stack of layers would collapse into a single linear function, which
is no better than the linear classifiers of lab 7.

<img src="https://raw.githubusercontent.com/soonho-eric/CSC371_Fa2026/main/ch12/figures/12_11.png" width="450"/>

The ones you will see most often:

* **Logistic (sigmoid)**, $\sigma(z) = 1/(1 + e^{-z})$, with outputs in $(0, 1)$. Used in lab 10.
  For large $|z|$ it is flat, so its gradient is nearly zero and learning slows down.
* **Hyperbolic tangent**, $\tanh(z)$, a rescaled logistic with outputs in $(-1, 1)$, centered at 0.
* **ReLU**, $\max(0, z)$. The default choice for hidden layers today. Its gradient is exactly 1
  for every $z > 0$, so it does not saturate the way the logistic does.
* **Softmax**, used on the **output** layer. It turns the $t$ net inputs into probabilities
  that add to 1:
  $$
  p_j = \frac{e^{z_j}}{\sum_{k=1}^{t} e^{z_k}} .
  $$

In [ ]:
z = torch.linspace(-5, 5, 200)

plt.plot(z, torch.sigmoid(z), lw=3, label='Logistic')
plt.plot(z, torch.tanh(z), lw=3, linestyle='--', label='Tanh')
plt.plot(z, torch.relu(z), lw=3, linestyle=':', label='ReLU')
plt.axhline(0, color='black', lw=0.5)
plt.ylim([-1.5, 1.5])
plt.xlabel('Net input $z$')
plt.ylabel(r'Activation $\sigma(z)$')
plt.legend(loc='lower right')
plt.show()

In [ ]:
z = torch.tensor([1.1, 2.0, 0.4])
p = torch.softmax(z, dim=0)

print('Probabilities:', p)
print('Sum:', p.sum())

## MNIST with torchvision

`torchvision.datasets` downloads common datasets. MNIST comes as 60,000 training images and
10,000 test images. We keep the **last 10,000 training images as a validation set**, so the
training set has 50,000. The next lab uses exactly the same split.

As in lab 10, each image is flattened into a row of 784 pixels. The pixels are scaled to
$[0, 1]$.

In [ ]:
image_path = './'
mnist_train = torchvision.datasets.MNIST(image_path, train=True, download=True)
mnist_test = torchvision.datasets.MNIST(image_path, train=False, download=True)

X = mnist_train.data.reshape(-1, 28*28).float() / 255.
y = mnist_train.targets
X_test = mnist_test.data.reshape(-1, 28*28).float() / 255.
y_test = mnist_test.targets

X_train, y_train = X[:50000], y[:50000]
X_valid, y_valid = X[50000:], y[50000:]

print('train:', X_train.shape, ' valid:', X_valid.shape, ' test:', X_test.shape)

In [ ]:
fig = plt.figure(figsize=(15, 3))
for i in range(10):
    ax = fig.add_subplot(1, 10, i+1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(f'{y_train[i]}', size=15)
plt.show()

A `DataLoader` shuffles the data and cuts it into minibatches. It replaces lab 10's
`minibatch_generator`.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)

torch.manual_seed(1)
batch_size = 100
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

x_batch, y_batch = next(iter(train_dl))
print(x_batch.shape, y_batch.shape)

## The lab 10 network, in PyTorch

`nn.Sequential` stacks layers in order. `nn.Linear(784, 50)` holds a $50 \times 784$ weight
matrix and 50 biases, the same as `weight_h` and `bias_h` in lab 10. This is the lab 10
network: 784 inputs, 50 sigmoid hidden units, 10 sigmoid outputs.

In [ ]:
model = nn.Sequential(
    nn.Linear(28*28, 50),
    nn.Sigmoid(),
    nn.Linear(50, 10),
    nn.Sigmoid()
)
print(model)
print(model[0].weight.shape, model[0].bias.shape)

Training has the same steps as lab 10, but each is now one line:

| lab 10 | PyTorch |
|---|---|
| `a_h, a_out = model.forward(X)` | `pred = model(x_batch)` |
| `mse_loss(...)` | `loss = loss_fn(pred, y_batch)` |
| `model.backward(...)` | `loss.backward()` |
| `model.weight_h -= learning_rate * ...` (four lines) | `optimizer.step()` |

PyTorch adds gradients up until they are reset, so each step ends with
`optimizer.zero_grad()`.

In [ ]:
def accuracy(model, X, y):
    with torch.no_grad():   # no gradients needed, just predictions
        pred = model(X.to(device))
    return (torch.argmax(pred, dim=1) == y.to(device)).float().mean().item()


def train(model, loss_fn, learning_rate, num_epochs, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    valid_acc_hist = []

    for epoch in range(num_epochs):
        start = time.time()
        for x_batch, y_batch in train_dl:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            pred = model(x_batch)             # 1. forward pass
            loss = loss_fn(pred, y_batch)     # 2. loss
            loss.backward()                   # 3. backpropagation
            optimizer.step()                  # 4. update the weights
            optimizer.zero_grad()             # 5. reset the gradients
        seconds = time.time() - start

        valid_acc_hist.append(accuracy(model, X_valid, y_valid))
        if verbose:
            print(f'Epoch {epoch+1:2d}  Loss {loss.item():.4f}  '
                  f'Valid acc {valid_acc_hist[-1]:.4f}  ({seconds:.1f} s)')
    return valid_acc_hist

In [ ]:
def mse_onehot(pred, y):
    return nn.MSELoss()(pred, nn.functional.one_hot(y, 10).float())

torch.manual_seed(1)
hist_lab10 = train(model, mse_onehot, learning_rate=1.0, num_epochs=10)

## ReLU and cross-entropy

Two changes:

* **ReLU** hidden units instead of sigmoid.
* **Cross-entropy loss** with softmax outputs instead of squared error. For a correct label
  $y$ the loss is $-\log p_y$, which punishes confident wrong answers much harder than squared
  error does. `nn.CrossEntropyLoss` applies the softmax itself, so the last layer has **no**
  activation, and it takes the integer labels directly.

These are often the default choice for this type of problem.

In [ ]:
torch.manual_seed(1)
model = nn.Sequential(
    nn.Linear(28*28, 50),
    nn.ReLU(),
    nn.Linear(50, 10)
)

hist_relu = train(model, nn.CrossEntropyLoss(), learning_rate=0.1, num_epochs=10)

In [ ]:
plt.plot(range(1, 11), hist_lab10, lw=3, marker='o', label='Sigmoid + squared error (lab 10)')
plt.plot(range(1, 11), hist_relu, lw=3, marker='o', label='ReLU + cross-entropy')
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.legend(loc='lower right')
plt.show()

## How many hidden units?

Now vary the width of the hidden layer. Each model trains for 10 epochs.

Try other sizes, more epochs, or a different learning rate.

In [ ]:
hidden_sizes = [2, 4, 16, 64, 256, 1024]
valid_acc = []
models = []

for num_hidden in hidden_sizes:
    torch.manual_seed(1)
    model = nn.Sequential(
        nn.Linear(28*28, num_hidden),
        nn.ReLU(),
        nn.Linear(num_hidden, 10)
    )
    hist = train(model, nn.CrossEntropyLoss(), learning_rate=0.1, num_epochs=10, verbose=False)
    valid_acc.append(hist[-1])
    print(f'{num_hidden:5d} hidden units: valid acc {hist[-1]:.4f}')
    models.append(model)

In [ ]:
plt.plot(hidden_sizes, valid_acc, lw=3, marker='o')
plt.xscale('log')
plt.xlabel('Number of hidden units')
plt.ylabel('Validation accuracy after 10 epochs')
plt.show()

## The test set

As in lab 7c, the test set is used once, at the end, on the model chosen with the validation
set.

In [ ]:
best = int(np.argmax(valid_acc))
print('Chosen:', hidden_sizes[best], 'hidden units')
print(f'Validation accuracy: {valid_acc[best]:.4f}')
print(f'Test accuracy:       {accuracy(models[best], X_test, y_test):.4f}')

### CPU or GPU?

One epoch of a small and a large model, timed on each device. The bigger the matrices, the
more a GPU helps. For a small model it may be no faster at all, because every minibatch also
has to be copied to the GPU.

In [ ]:
devices = ['cpu', 'cuda'] if torch.cuda.is_available() else ['cpu']

for num_hidden in [64, 1024]:
    for d in devices:
        model = nn.Sequential(nn.Linear(28*28, num_hidden), nn.ReLU(),
                              nn.Linear(num_hidden, 10)).to(d)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
        loss_fn = nn.CrossEntropyLoss()

        start = time.time()
        for x_batch, y_batch in train_dl:
            x_batch, y_batch = x_batch.to(d), y_batch.to(d)
            loss = loss_fn(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        print(f'{num_hidden:5d} hidden units on {d}: {time.time() - start:.2f} s per epoch')

## Next lab

Every model here treats an image as a list of 784 unrelated numbers: shuffle the pixels the
same way in every image and it would learn just as well. The next lab introduces
**convolutional neural networks**, which use the fact that nearby pixels belong together, and
compares them with the linear classifiers, decision trees, and the networks from this lab, on
this same split of MNIST.